In [25]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torchvision
from PIL import Image

train_data = np.load('Numarul4-train/2_4_train.npz', allow_pickle=True)
x_train, y_train = train_data['x'], train_data['y']
print(x_train.shape, y_train.shape)

test_data = np.load('Numarul4-test/2_4_test.npz', allow_pickle=True)
x_test, y_test = test_data['x'], test_data['y']
print(x_test.shape, y_test.shape)

print(x_train[0].shape)

N_CLASSES = len(np.unique(y_train))
print(f"N_CLASSES: {N_CLASSES}")

dim_W = [img.shape[0] for img in x_train]
dim_H = [img.shape[1] for img in x_train]

print(min(dim_H), max(dim_H))
print(min(dim_W), max(dim_W))

H = W = 48

(27705,) (27705,)
(9541,) (9541,)
(63, 46, 3)
N_CLASSES: 5
32 64
32 64


In [26]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.images = x
        self.labels = y

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx])
        label = self.labels[idx]

        image = image.resize(size=(H, W))
        image = torch.as_tensor(data=np.array(image), dtype=torch.float32)
        image = image / 255.0
        image = torch.permute(image, dims=(2, 0, 1))

        label = torch.as_tensor(data=label, dtype=torch.int64)

        return image, label


train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [27]:
class Model(nn.Module):
    def __init__(self, act1, act2, in_channels=3, n_classes=N_CLASSES):
        super().__init__()
        self.act1 = act1
        self.act2 = act2

        # IN -> (3, 48, 48)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=48, kernel_size=3, padding=1) # -> (48, 48, 48)
        self.pool1 = nn.MaxPool2d(kernel_size=2) # -> (48, 24, 24)

        self.conv2 = nn.Conv2d(in_channels=48, out_channels=32, kernel_size=3, padding=1) # -> (32, 24, 24)
        self.pool2 = nn.MaxPool2d(kernel_size=2) # -> (32, 12, 12)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=24, kernel_size=3, padding=1) # -> (24, 12, 12)
        self.pool3 = nn.MaxPool2d(kernel_size=2) # -> (24, 6, 6)

        self.fc1 = nn.Linear(in_features=24 * 6 * 6, out_features=288)
        self.drop1 = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(in_features=288, out_features=128)
        self.drop2 = nn.Dropout(p=0.5)
        self.fc3 = nn.Linear(in_features=128, out_features=n_classes)


    def forward(self, x):
        x = self.pool1(self.act1(self.conv1(x)))
        x = self.pool2(self.act2(self.conv2(x)))
        x = self.pool3(self.act1(self.conv3(x)))
        
        x = x.flatten(1)
        x = self.drop1(self.act1(self.fc1(x)))
        x = self.drop2(self.act1(self.fc2(x)))
        x = self.fc3(x)

        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(act1=nn.functional.relu, act2=nn.functional.leaky_relu).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
epochs = 20


def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total


def test(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            total_loss += loss.item()
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return total_loss / len(loader), correct / total


best = 0
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = test(model, test_loader, loss_fn, device)
    print(f"Epoch {epoch}: train loss={train_loss:.4f}, acc={train_acc:.4f} | test loss={test_loss:.4f}, acc={test_acc:.4f}")
    if best < test_acc:
        best = test_acc  
        torch.save(model, 'model.pt')


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return correct / total


model = torch.load('model.pt', weights_only=False)
test_acc = evaluate(model, test_loader, device)
print(f'{test_acc:.4f}')

Epoch 0: train loss=1.4334, acc=0.3554 | test loss=0.6768, acc=0.7987
Epoch 1: train loss=0.4925, acc=0.8345 | test loss=0.3567, acc=0.8929
Epoch 2: train loss=0.3374, acc=0.8919 | test loss=0.2929, acc=0.9126
Epoch 3: train loss=0.2825, acc=0.9107 | test loss=0.2819, acc=0.9132
Epoch 4: train loss=0.2510, acc=0.9206 | test loss=0.2608, acc=0.9226
Epoch 5: train loss=0.2312, acc=0.9276 | test loss=0.2634, acc=0.9218
Epoch 6: train loss=0.2087, acc=0.9335 | test loss=0.2515, acc=0.9267
Epoch 7: train loss=0.1961, acc=0.9384 | test loss=0.2503, acc=0.9287
Epoch 8: train loss=0.1826, acc=0.9430 | test loss=0.2753, acc=0.9193
Epoch 9: train loss=0.1700, acc=0.9450 | test loss=0.2542, acc=0.9279
Epoch 10: train loss=0.1618, acc=0.9481 | test loss=0.2395, acc=0.9355
Epoch 11: train loss=0.1481, acc=0.9526 | test loss=0.2441, acc=0.9326
Epoch 12: train loss=0.1360, acc=0.9567 | test loss=0.2576, acc=0.9325
Epoch 13: train loss=0.1339, acc=0.9568 | test loss=0.2514, acc=0.9346
Epoch 14: train 